In [1]:
# imports
import os, av, numpy as np, torch
import torch.nn as nn
import torch.nn.functional as F
import timm
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler

# constants
SEED = 42
FLOW_CACHE_DIR = "flow_cache"
GESTURE_MAP = {'G1':0,'G2':1,'G3':2,'G4':3,'G5':4,
               'G6':5,'G8':6,'G9':7,'G10':8,'G11':9}
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

In [2]:
import torch.nn as nn
from torchvision.models import resnet50, ResNet50_Weights

class SurgicalTransformer(nn.Module):
    def __init__(self, n_classes=10, clip_len=8, feat_dim=512, 
                n_heads=8, n_layers=4, dropout=0.1):
        super().__init__()
        self.clip_len = clip_len

        # resnet50 but strip the final layer to output 2048-dim features instead of class scores
        resnet = resnet50(weights=ResNet50_Weights.DEFAULT)
        self.frame_encoder = nn.Sequential(*list(resnet.children())[:-1])

        # project 2048 -> 512 smaller and faster transformer input
        self.proj = nn.Linear(2048, feat_dim)

        # temporal transformer
        # this accumulates the summary of the entire clip
        self.cls_token = nn.Parameter(torch.randn(1, 1, feat_dim))

        # give it the postional embedding as well
        self.pos_embed = nn.Parameter(torch.randn(1, clip_len+1, feat_dim))

        # 4 layer transformer encoder... each frame attends to every other frame
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=feat_dim, nhead=n_heads,
            dim_feedforward=feat_dim * 4,
            dropout=dropout, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, n_layers)
        self.norm = nn.LayerNorm(feat_dim)

        # classifier
        self.head = nn.Linear(feat_dim, n_classes)

    
    def forward(self, x):
        # x is (B, T, C, H, W) - batch of clips
        # returns (B, n_classes) - logits per clip

        B, T, C, H, W = x.shape

        # extract features from every fram independently
        # reshape so all frames are processed at once

        x = x.view(B*T, C, H, W)
        x = self.frame_encoder(x)
        x = x.view(B*T, -1)
        x = self.proj(x)
        x = x.view(B, T, -1)

        # prepend the cls token
        cls = self.cls_token.expand(B, -1, -1)
        x = torch.cat([cls, x], dim=1)

        # add pos embeddings
        x = x + self.pos_embed

        # transformer
        x = self.transformer(x)
        x = self.norm(x)

        # cls token at pos 0 
        clip_repr = x[:, 0, :]

        return self.head(clip_repr)

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
model2 = SurgicalTransformer(n_classes=10, clip_len=8).to(device)

criterion2 = nn.CrossEntropyLoss()
optimizer2 = torch.optim.Adam(model2.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler2 = torch.optim.lr_scheduler.StepLR(optimizer2, step_size=5, gamma=0.5)

total = sum(p.numel() for p in model2.parameters() if p.requires_grad)
print(f"Trainable params: {total:,}")
print(f"Device: {device}")


Trainable params: 37,177,930
Device: mps


In [3]:
import timm
import torch.nn as nn
import torch.nn.functional as F

class SAISModel(nn.Module):
    def __init__(self, n_classes = 10, clip_len=8, feat_dim=384,
                 proj_dim=256, n_heads=6, n_layers=4, dropout=0.1):
        super().__init__()
        self.clip_len = clip_len

        # RGB stream - frozen DINO-ViT-Small
        self.rgb_vit = timm.create_model(
            'vit_small_patch16_224.dino',
            pretrained=True, num_classes=0
        )
        for p in self.rgb_vit.parameters():
            p.requires_grad = False

        # FLOW stream - frozen DINO-ViT-Small
        # flow has 2 channels not 3 so we poject 2ch -> 3ch with a small learned conv before ViT
        self.flow_proj = nn.Conv2d(2, 3, kernel_size=1) # learned projection - SAIS does this differently
        self.flow_vit = timm.create_model(
            'vit_small_patch16_224.dino',
            pretrained=True, num_classes=0
        )
        for p in self.flow_vit.parameters():
            p.requires_grad = False


        # RGB transformer
        self.rgb_cls   = nn.Parameter(torch.randn(1, 1, feat_dim))
        self.rgb_pos   = nn.Parameter(torch.randn(1, clip_len + 1, feat_dim))
        rgb_layer      = nn.TransformerEncoderLayer(
            d_model=feat_dim, nhead=n_heads,
            dim_feedforward=feat_dim * 4,
            dropout=dropout, batch_first=True
        )
        self.rgb_transformer = nn.TransformerEncoder(rgb_layer, n_layers)
        self.rgb_norm        = nn.LayerNorm(feat_dim)

        # Flow transformer - separate weights, same architecture
        self.flow_cls  = nn.Parameter(torch.randn(1, 1, feat_dim))
        self.flow_pos  = nn.Parameter(torch.randn(1, clip_len + 1, feat_dim))
        flow_layer     = nn.TransformerEncoderLayer(
            d_model=feat_dim, nhead=n_heads,
            dim_feedforward=feat_dim * 4,
            dropout=dropout, batch_first=True
        )
        self.flow_transformer = nn.TransformerEncoder(flow_layer, n_layers)
        self.flow_norm        = nn.LayerNorm(feat_dim)

        # proj head
        self.proj_head = nn.Sequential(
            nn.Linear(feat_dim * 2, proj_dim),
            nn.ReLU(),
            nn.Linear(proj_dim, proj_dim)
        )

        # prototype classifier
        self.prototypes = nn.Parameter(torch.randn(n_classes, proj_dim))
        self.temperature = 0.07

    def encode_stream(self, x, vit):
        # encode T frames through frozen Vit independently
        # x:   (B, T, C, H, W)
        # out: (B, T, feat_dim)
        B, T, C, H, W = x.shape
        x = x.view(B*T, C, H, W)
        with torch.no_grad():
            x = vit(x)          # (B*T, feat_dim
        return x.view(B, T, -1)     # (B, T, feat_dim)
    

    def temporal_encode(self, x, cls_token, pos_embed, transformer, norm):
        # run temporal transformeer on frame features
        # x:   (B, T, feat_dim)
        # out: (B, feat_dim) — CLS token

        B = x.shape[0]
        cls = cls_token.expand(B, -1, -1)
        x   = torch.cat([cls, x], dim=1)   # (B, T+1, feat_dim)
        x   = x + pos_embed
        x   = transformer(x)
        x   = norm(x)
        return x[:, 0, :]     # cls token (B, feat_dim)
    
    def forward(self, rgb, flow, labels=None):
        # rgb:    (B, T, 3, H, W)
        # flow:   (B, T, 2, H, W)
        # labels: (B,) during training, None at inference
        

        B, T, _, H, W = flow.shape

        # proj the 2ch to 3ch
        flow_3ch = self.flow_proj(flow.view(B*T, 2, H, W)) # (B*T, 3, H, W)
        flow_3ch = flow_3ch.view(B, T, 3, H, W)

        # extract rgb and flow feat
        rgb_feat = self.encode_stream(rgb, self.rgb_vit) # (B, T, 384)
        flow_feat = self.encode_stream(flow_3ch, self.flow_vit) # (B, T, 384)

        # separate temporal transformers per stream
        h_rgb  = self.temporal_encode(
            rgb_feat,  self.rgb_cls,  self.rgb_pos,
            self.rgb_transformer,  self.rgb_norm
        )  # (B, 384)
        h_flow = self.temporal_encode(
            flow_feat, self.flow_cls, self.flow_pos,
            self.flow_transformer, self.flow_norm
        )  # (B, 384)

        # fuse by concat - proj head later learns to combine - future improv to same as SAIS (likely a HSV visualization)
        fused = torch.cat([h_rgb, h_flow], dim=-1)  # (B, 768)

        # proj + proto classify
        h = self.proj_head(fused)       # (B, proj_dim)
        h_norm = F.normalize(h, dim=1)
        p_norm = F.normalize(self.prototypes, dim=1)
        logits = h_norm @ p_norm.T / self.temperature

        if labels is not None:
            loss = F.cross_entropy(logits, labels)
            return loss, logits
        return logits
    
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
torch.manual_seed(SEED)
model4 = SAISModel(n_classes=10, clip_len=8).to(device)

trainable4 = [p for p in model4.parameters() if p.requires_grad]
optimizer4 = torch.optim.Adam(trainable4, lr=1e-4, weight_decay=1e-4)
scheduler4 = torch.optim.lr_scheduler.StepLR(optimizer4, step_size=5, gamma=0.5)

total = sum(p.numel() for p in model4.parameters() if p.requires_grad)
print(f"Trainable params: {total:,}  (both ViTs frozen)")
print(f"Device: {device}")

Trainable params: 14,470,153  (both ViTs frozen)
Device: mps


In [9]:
# ============================================================
# SKILL EVALUATION FROM GESTURE-TRAINED EMBEDDINGS
# Tests whether gesture-trained models implicitly learned skill
# No retraining on skill — purely extracting learned representations

import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# ── Step 1: Parse skill labels (N=0, E=1, skip I) ───────────────────────────

def parse_skill_labels(task='Suturing'):
    meta_path = f"JIGSAWS/{task}/meta_file_{task}.txt"
    skill_map = {}
    with open(meta_path) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 3:
                continue
            trial = parts[0].replace(f'{task}_', '')
            level = parts[1]
            grs   = int(parts[2])
            if level == 'N':
                skill_map[trial] = (0, grs)   # (skill_label, grs_score)
            elif level == 'E':
                skill_map[trial] = (1, grs)
            # skip I
    return skill_map

skill_map = parse_skill_labels('Suturing')
print("Skill labels (N=0, E=1):")
for trial, (label, grs) in sorted(skill_map.items()):
    print(f"  {trial}: {'Expert' if label==1 else 'Novice'} | GRS: {grs}")

novice_trials = [t for t,(l,_) in skill_map.items() if l==0]
expert_trials  = [t for t,(l,_) in skill_map.items() if l==1]
print(f"\nNovice trials: {len(novice_trials)}")
print(f"Expert trials: {len(expert_trials)}")


# ── Step 2: Dataset returning clips + skill labels ───────────────────────────

class SkillEvalDataset(Dataset):
    """
    Returns gesture clips with skill labels.
    Model sees NO skill labels during training —
    we use them only for evaluation here.
    use_flow=True  → returns (rgb, flow, skill_label) for Stage 4
    use_flow=False → returns (rgb, skill_label) for Stage 2
    """
    def __init__(self, task='Suturing', skill_map=None,
                 clip_len=8, use_flow=False):
        self.clip_len  = clip_len
        self.use_flow  = use_flow
        self.clips     = []   # (video_path, start, end, skill_label, key)

        trans_dir = f"JIGSAWS/{task}/transcriptions/"
        video_dir = f"JIGSAWS/{task}/video/"

        for fname in sorted(os.listdir(trans_dir)):
            if not fname.endswith('.txt'):
                continue
            trial = fname.replace(f'{task}_', '').replace('.txt', '')
            if trial not in skill_map:
                continue   # skip Intermediate

            video_path  = os.path.join(video_dir,
                                        f"{task}_{trial}_capture1.avi")
            if not os.path.exists(video_path):
                continue

            skill_label = skill_map[trial][0]   # 0=Novice, 1=Expert

            with open(os.path.join(trans_dir, fname)) as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) != 3:
                        continue
                    start, end, gesture = int(parts[0]), int(parts[1]), parts[2]
                    if gesture in GESTURE_MAP and (end - start) >= clip_len:
                        key = f"{task}_{trial}_{start}_{end}.npy"
                        self.clips.append((video_path, start, end,
                                           skill_label, key))

        self.rgb_transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Resize((224, 224), antialias=True),
            transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
        ])
        print(f"  Skill eval dataset: {len(self.clips)} clips "
              f"({sum(1 for c in self.clips if c[3]==0)} Novice, "
              f"{sum(1 for c in self.clips if c[3]==1)} Expert)")

    def _load_rgb(self, video_path, start, end):
        frame_ids = set(np.linspace(start, end,
                                    self.clip_len, dtype=int).tolist())
        frames = []
        container = av.open(video_path)
        for i, frame in enumerate(container.decode(video=0)):
            if i in frame_ids:
                frames.append(self.rgb_transform(
                    frame.to_ndarray(format='rgb24')))
            if len(frames) == self.clip_len:
                break
        container.close()
        while len(frames) < self.clip_len:
            frames.append(frames[-1])
        return torch.stack(frames)

    def _load_flow(self, key):
        flow = np.load(os.path.join(FLOW_CACHE_DIR, key))
        flow = torch.tensor(flow, dtype=torch.float32)
        flow = flow / (flow.abs().max() + 1e-6)
        last = flow[-1:].clone()
        flow = torch.cat([flow, last], dim=0)
        return flow

    def __len__(self): return len(self.clips)

    def __getitem__(self, idx):
        video_path, start, end, label, key = self.clips[idx]
        rgb = self._load_rgb(video_path, start, end)
        if self.use_flow:
            flow = self._load_flow(key)
            return rgb, flow, label
        return rgb, label


# ── Step 3: Embedding extraction functions ───────────────────────────────────

def extract_stage2_embeddings(model2, dataloader, device):
    """
    Extract CLS token representations from Stage 2 model.
    Shape: (N_clips, 512)
    Does NOT use the classifier head — raw Transformer output.
    """
    model2.eval()
    all_embs   = []
    all_labels = []

    with torch.no_grad():
        for clips, labels in dataloader:
            clips = clips.to(device)
            B, T, C, H, W = clips.shape

            # ResNet frame encoding
            x = clips.view(B*T, C, H, W)
            x = model2.frame_encoder(x)          # (B*T, 2048, 1, 1)
            x = x.squeeze(-1).squeeze(-1)         # (B*T, 2048)
            x = model2.proj(x)                    # (B*T, 512)
            x = x.view(B, T, -1)                  # (B, T, 512)

            # Temporal Transformer
            cls = model2.cls_token.expand(B, -1, -1)
            x   = torch.cat([cls, x], dim=1)      # (B, T+1, 512)
            x   = x + model2.pos_embed
            x   = model2.transformer(x)
            x   = model2.norm(x)
            emb = x[:, 0, :]                       # CLS token (B, 512)

            all_embs.append(emb.cpu())
            all_labels.extend(labels.numpy())

    X = torch.cat(all_embs).numpy()    # (N_clips, 512)
    y = np.array(all_labels)           # (N_clips,)
    print(f"  Extracted {X.shape[0]} embeddings, dim={X.shape[1]}")
    return X, y


def extract_stage4_embeddings(model4, dataloader, device):
    """
    Extract fused CLS token representations from Stage 4 model.
    Inlines all operations — no helper method assumptions.
    Shape: (N_clips, 768)
    """
    model4.eval()
    all_embs   = []
    all_labels = []

    with torch.no_grad():
        for rgb, flow, labels in dataloader:
            rgb, flow = rgb.to(device), flow.to(device)
            B, T, _, H, W = flow.shape

            # ── flow 2ch → 3ch ──────────────────────────────────────
            flow_3ch = model4.flow_proj(flow.view(B*T, 2, H, W))
            flow_3ch = flow_3ch.view(B, T, 3, H, W)

            # ── RGB stream: frozen ViT ───────────────────────────────
            rgb_in   = rgb.view(B*T, 3, H, W)
            rgb_feat = model4.rgb_vit(rgb_in)        # (B*T, 384)
            rgb_feat = rgb_feat.view(B, T, -1)        # (B, T, 384)

            # ── Flow stream: frozen ViT ──────────────────────────────
            flo_in    = flow_3ch.view(B*T, 3, H, W)
            flow_feat = model4.flow_vit(flo_in)       # (B*T, 384)
            flow_feat = flow_feat.view(B, T, -1)       # (B, T, 384)

            # ── RGB temporal transformer ─────────────────────────────
            cls  = model4.rgb_cls.expand(B, -1, -1)
            x    = torch.cat([cls, rgb_feat], dim=1)   # (B, T+1, 384)
            x    = x + model4.rgb_pos
            x    = model4.rgb_transformer(x)
            x    = model4.rgb_norm(x)
            h_rgb = x[:, 0, :]                          # (B, 384)

            # ── Flow temporal transformer ────────────────────────────
            cls  = model4.flow_cls.expand(B, -1, -1)
            x    = torch.cat([cls, flow_feat], dim=1)  # (B, T+1, 384)
            x    = x + model4.flow_pos
            x    = model4.flow_transformer(x)
            x    = model4.flow_norm(x)
            h_flow = x[:, 0, :]                         # (B, 384)

            # ── fuse BEFORE projection head ──────────────────────────
            emb = torch.cat([h_rgb, h_flow], dim=-1)   # (B, 768)

            all_embs.append(emb.cpu())
            all_labels.extend(labels.numpy())

    X = torch.cat(all_embs).numpy()
    y = np.array(all_labels)
    print(f"  Extracted {X.shape[0]} embeddings, dim={X.shape[1]}")
    return X, y


# ── Step 4: AUC evaluation with cross-validation ─────────────────────────────

def compute_skill_auc(X, y, n_splits=5):
    """
    Cross-validated AUC using logistic regression.
    More reliable than a single train/test split given small dataset.
    Returns mean AUC and std across folds.
    """
    # normalize embeddings — logistic regression needs this
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    aucs = []
    cv   = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

    for fold, (train_idx, test_idx) in enumerate(cv.split(X_scaled, y)):
        X_tr, X_te = X_scaled[train_idx], X_scaled[test_idx]
        y_tr, y_te = y[train_idx], y[test_idx]

        if len(np.unique(y_te)) < 2:
            continue   # skip fold if only one class in test

        clf = LogisticRegression(max_iter=1000, C=1.0)
        clf.fit(X_tr, y_tr)
        probs = clf.predict_proba(X_te)[:, 1]
        fold_auc = roc_auc_score(y_te, probs)
        aucs.append(fold_auc)

    mean_auc = np.mean(aucs)
    std_auc  = np.std(aucs)
    return mean_auc, std_auc


# ── Step 5: Run the full evaluation ─────────────────────────────────────────

print("\n" + "="*60)
print("SKILL EVALUATION FROM GESTURE-TRAINED EMBEDDINGS")
print("="*60)
print("No skill labels used during training.")
print("Testing whether gesture representations encode skill.")
print("="*60)

# build skill evaluation dataset
print("\nBuilding skill evaluation datasets...")
skill_ds_s2 = SkillEvalDataset('Suturing', skill_map=skill_map,
                                clip_len=8, use_flow=False)
skill_ds_s4 = SkillEvalDataset('Suturing', skill_map=skill_map,
                                clip_len=8, use_flow=True)

skill_loader_s2 = DataLoader(skill_ds_s2, batch_size=8,
                              shuffle=False, num_workers=0)
skill_loader_s4 = DataLoader(skill_ds_s4, batch_size=8,
                              shuffle=False, num_workers=0)
                              
# load Stage 2 model
print("\nLoading Stage 2 model...")
ckpt2  = torch.load('stage2_combined_best.pt', map_location=device)
model2 = SurgicalTransformer(n_classes=10, clip_len=8).to(device)
model2.load_state_dict(ckpt2)
model2.eval()
print("  Stage 2 loaded successfully")

# load Stage 4 model
print("\nLoading Stage 4 model...")
ckpt4  = torch.load('stage4_combined_best.pt', map_location=device)
model4_skill = SAISModel(n_classes=10, clip_len=8).to(device)
model4_skill.load_state_dict(ckpt4['model_state_dict'])
model4_skill.eval()
print(f"  Stage 4 best F1 was: {ckpt4['best_f1']:.3f}")

# extract embeddings
print("\nExtracting Stage 2 embeddings...")
X2, y2 = extract_stage2_embeddings(model2, skill_loader_s2, device)

print("\nExtracting Stage 4 embeddings...")
X4, y4 = extract_stage4_embeddings(model4_skill, skill_loader_s4, device)

# compute AUC
print("\nComputing skill AUC (5-fold cross-validation)...")
auc2_mean, auc2_std = compute_skill_auc(X2, y2, n_splits=5)
auc4_mean, auc4_std = compute_skill_auc(X4, y4, n_splits=5)

# results
print("\n" + "="*60)
print("RESULTS — Skill Detection from Gesture Representations")
print("="*60)
print(f"  {'Model':<30} {'AUC':>8} {'Std':>8}")
print(f"  {'-'*48}")
print(f"  {'Stage 2 (ResNet + Transformer)':<30} {auc2_mean:>8.3f} {auc2_std:>8.3f}")
print(f"  {'Stage 4 (DINO + Flow + 2-stream)':<30} {auc4_mean:>8.3f} {auc4_std:>8.3f}")


Skill labels (N=0, E=1):
  B001: Novice | GRS: 13
  B002: Novice | GRS: 17
  B003: Novice | GRS: 14
  B004: Novice | GRS: 10
  B005: Novice | GRS: 12
  D001: Expert | GRS: 14
  D002: Expert | GRS: 18
  D003: Expert | GRS: 14
  D004: Expert | GRS: 8
  D005: Expert | GRS: 15
  E001: Expert | GRS: 17
  E002: Expert | GRS: 20
  E003: Expert | GRS: 19
  E004: Expert | GRS: 19
  E005: Expert | GRS: 19
  G001: Novice | GRS: 13
  G002: Novice | GRS: 18
  G003: Novice | GRS: 13
  G004: Novice | GRS: 21
  G005: Novice | GRS: 23
  H001: Novice | GRS: 14
  H003: Novice | GRS: 25
  H004: Novice | GRS: 19
  H005: Novice | GRS: 21
  I001: Novice | GRS: 17
  I002: Novice | GRS: 23
  I003: Novice | GRS: 17
  I004: Novice | GRS: 23
  I005: Novice | GRS: 19

Novice trials: 19
Expert trials: 10

SKILL EVALUATION FROM GESTURE-TRAINED EMBEDDINGS
No skill labels used during training.
Testing whether gesture representations encode skill.

Building skill evaluation datasets...
  Skill eval dataset: 607 clips (